##### Syllabic nuclei computing

In [6]:
import math
from pathlib import Path
import re

import numpy as np
import parselmouth
from praatio import tgio

In [7]:
ROOT = Path("/Users/moanason/Downloads/Data_ANA")

# intensity / pitch parameters for syllable detection
INT_TIME_STEP = 0.008   # slightly smaller 10 ms
PITCH_FLOOR = 75
PITCH_CEIL  = 500 

MIN_DIP_DB  = 1.0 
PEAK_PERCENTILE = 75

In [8]:
def parse_combined_tg_name(name: str):
    pattern = re.compile(r"ref_s(\d+)_N([CS])(\d*)_processed", re.IGNORECASE)
    m = pattern.search(name)
    if not m:
        return None
    sess_num = int(m.group(1))
    cs_flag  = m.group(2).upper()   
    rep      = m.group(3) or "1" 
    return sess_num, cs_flag, rep


def mono_audio_path(sess_num: int, spk: str, cs_flag: str, rep: str) -> Path:
    sess_str = f"{sess_num:02d}"
    cond_part = f"N{cs_flag}{rep}_processed"
    fname = f"p142_s{sess_str}_{spk}_{cond_part}.wav"
    path = ROOT / fname
    if not path.is_file():
        raise FileNotFoundError(f"error path: {path}")
    return path


In [9]:
def detect_syllable_nuclei(
    sound: parselmouth.Sound, # for tuning to catch more for swissgerman
    time_step=0.008,
    pitch_floor=75,
    pitch_ceiling=500,
    peak_percentile=70,
    peak_margin_db=1.0,
    min_dip_db=0.5, 
    min_interval=0.015,
):
    intensity = sound.to_intensity(time_step=time_step)
    times = intensity.xs()
    vals = intensity.values[0]

    vals = np.where(np.isfinite(vals), vals, np.nan)
    finite_vals = vals[np.isfinite(vals)]
    if finite_vals.size == 0:
        return []

    thr = np.percentile(finite_vals, peak_percentile) - peak_margin_db

    pitch = sound.to_pitch(
        time_step=time_step,
        pitch_floor=pitch_floor,
        pitch_ceiling=pitch_ceiling,
    )

    syllable_points = []
    last_t = -1e9

    for i in range(1, len(times) - 1):
        v = vals[i]
        if not np.isfinite(v):
            continue

        # local maximum
        if not (v > vals[i - 1] and v >= vals[i + 1]):
            continue

        if v < thr:
            continue

        left_dip = v - vals[i - 1] if np.isfinite(vals[i - 1]) else 0
        right_dip = v - vals[i + 1] if np.isfinite(vals[i + 1]) else 0
        if left_dip < min_dip_db and right_dip < min_dip_db:
            continue

        t = float(times[i])
        if t - last_t < min_interval:
            continue

        f0 = pitch.get_value_at_time(t)
        if math.isnan(f0) or f0 <= 0:
            continue

        syllable_points.append((t, "σ"))
        last_t = t

    return syllable_points


In [10]:
tg_paths = sorted(ROOT.glob("ref_s*_N*_processed.TextGrid"))

print(f"Found {len(tg_paths)} combined TextGrids.")

for tg_path in tg_paths:
    info = parse_combined_tg_name(tg_path.name)
    if info is None:
        print("Skipping (name pattern mismatch):", tg_path.name)
        continue

    sess_num, cs_flag, rep = info
    print(f"\nProcessing {tg_path.name} (session {sess_num}, {cs_flag}{rep})")

    # Load mono audios for A and B
    path_A = mono_audio_path(sess_num, "A", cs_flag, rep)
    path_B = mono_audio_path(sess_num, "B", cs_flag, rep)

    snd_A = parselmouth.Sound(str(path_A))
    snd_B = parselmouth.Sound(str(path_B))

    # Detect syllable nuclei
    syl_A = detect_syllable_nuclei(snd_A)
    syl_B = detect_syllable_nuclei(snd_B)

    print(f"Speaker A: {len(syl_A)} syllable nuclei")
    print(f"Speaker B: {len(syl_B)} syllable nuclei")

    tg = tgio.openTextgrid(str(tg_path))

    if "Transcribe_A" in tg.tierNameList:
        tg.removeTier("Transcribe_A")
    if "Transcribe_B" in tg.tierNameList:
        tg.removeTier("Transcribe_B")

    dur_A = float(snd_A.duration)
    dur_B = float(snd_B.duration)
    max_time = max(dur_A, dur_B)
    min_time = 0.0
    tier_A = tgio.PointTier("Transcribe_A", syl_A,
                            min_time, max_time)
    tier_B = tgio.PointTier("Transcribe_B", syl_B,
                            min_time, max_time)

    tg.addTier(tier_A)
    tg.addTier(tier_B)

    tg.save(str(tg_path))
    print(f"Updated TextGrid (added Transcribe_A/B): {tg_path.name}")


Found 30 combined TextGrids.

Processing ref_s05_NC1_processed.TextGrid (session 5, C1)
Speaker A: 764 syllable nuclei
Speaker B: 1792 syllable nuclei
Updated TextGrid (added Transcribe_A/B): ref_s05_NC1_processed.TextGrid

Processing ref_s05_NC2_processed.TextGrid (session 5, C2)
Speaker A: 821 syllable nuclei
Speaker B: 1600 syllable nuclei
Updated TextGrid (added Transcribe_A/B): ref_s05_NC2_processed.TextGrid

Processing ref_s06_NC1_processed.TextGrid (session 6, C1)
Speaker A: 987 syllable nuclei
Speaker B: 1598 syllable nuclei
Updated TextGrid (added Transcribe_A/B): ref_s06_NC1_processed.TextGrid

Processing ref_s06_NC2_processed.TextGrid (session 6, C2)
Speaker A: 1141 syllable nuclei
Speaker B: 693 syllable nuclei
Updated TextGrid (added Transcribe_A/B): ref_s06_NC2_processed.TextGrid

Processing ref_s07_NC1_processed.TextGrid (session 7, C1)
Speaker A: 751 syllable nuclei
Speaker B: 1961 syllable nuclei
Updated TextGrid (added Transcribe_A/B): ref_s07_NC1_processed.TextGrid

